# HNSW Search Evaluation

Notebook ini mengevaluasi retrieval HNSW dari Qdrant terhadap `Ground truth - System.csv`. Karena response search di `src/search_engine.py` digroup berdasarkan agreement, evaluasi di sini melakukan flatten kembali ke level artikel: 10 agreement teratas x top 3 artikel per agreement = maksimum 30 artikel per query.

Grid evaluasi:
1. Collection hasil indexing: `m8/m16/m32` dan `ef_construct 100/200/400`
2. Parameter search: `hnsw_ef = 32, 64, 128, 256`
3. Limit pengembalian Qdrant: `10, 20, 30, 50, 100`

Catatan evaluasi: default notebook ini memakai **GT-pool filtering**, yaitu artikel hasil search yang tidak ada di ground truth dibuang sebelum menghitung NDCG@30. Ini sesuai kebutuhan pengujian terhadap artikel yang sudah diberi relevance. Jika ingin memberi penalti pada artikel yang tidak ada di GT, ubah `FILTER_TO_GT_POOL = False`.

In [40]:
from __future__ import annotations

import re
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

# Path dibuat aman baik notebook dijalankan dari root repo maupun dari folder evaluation.
CWD = Path.cwd().resolve()
if (CWD / "Ground truth - System.csv").exists():
    EVAL_DIR = CWD
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD
    EVAL_DIR = PROJECT_ROOT / "evaluation"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import importlib
import src.config as config_module

# Reload dibutuhkan kalau config.py baru diedit tetapi kernel notebook belum direstart.
config_module = importlib.reload(config_module)
settings = config_module.settings

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EVAL_DIR:", EVAL_DIR)
print("Config file:", config_module.__file__)

PROJECT_ROOT: /home/vinik/Github/vector-search-thesis
EVAL_DIR: /home/vinik/Github/vector-search-thesis/evaluation
Config file: /home/vinik/Github/vector-search-thesis/src/config.py


In [41]:
GT_PATH = EVAL_DIR / "Ground truth - System.csv"
OUTPUT_DIR = EVAL_DIR / "hnsw_eval_results"

COLLECTION_CONFIGS = [
    {
        "collection_id": "m8_ef100",
        "collection_name": settings.COLLECTION_HYBRID_M8_EF100,
        "m": 8,
        "ef_construct": 100,
    },
    {
        "collection_id": "m8_ef200",
        "collection_name": settings.COLLECTION_HYBRID_M8_EF200,
        "m": 8,
        "ef_construct": 200,
    },
    {
        "collection_id": "m8_ef400",
        "collection_name": settings.COLLECTION_HYBRID_M8_EF400,
        "m": 8,
        "ef_construct": 400,
    },
    {
        "collection_id": "m16_ef100",
        "collection_name": settings.COLLECTION_HYBRID_M16_EF100,
        "m": 16,
        "ef_construct": 100,
    },
    {
        "collection_id": "m16_ef200",
        "collection_name": settings.COLLECTION_HYBRID_M16_EF200,
        "m": 16,
        "ef_construct": 200,
    },
    {
        "collection_id": "m16_ef400",
        "collection_name": settings.COLLECTION_HYBRID_M16_EF400,
        "m": 16,
        "ef_construct": 400,
    },
    {
        "collection_id": "m32_ef100",
        "collection_name": settings.COLLECTION_HYBRID_M32_EF100,
        "m": 32,
        "ef_construct": 100,
    },
    {
        "collection_id": "m32_ef200",
        "collection_name": settings.COLLECTION_HYBRID_M32_EF200,
        "m": 32,
        "ef_construct": 200,
    },
    {
        "collection_id": "m32_ef400",
        "collection_name": settings.COLLECTION_HYBRID_M32_EF400,
        "m": 32,
        "ef_construct": 400,
    },
]

EF_VALUES = [16, 32, 64, 128, 256]
QDRANT_LIMIT_VALUES = list(range(10, 101, 10))
K = 30
AGREEMENT_LIMIT = 10
TOP_ARTICLES_PER_AGREEMENT = 3
TIMING_RUNS = 25
THRESHOLD = 0.0

# True = hanya hitung artikel yang ada di GT pool. False = artikel unjudged dihitung sebagai relevance 0.
FILTER_TO_GT_POOL = True

# Mode search per unit GT.
# negation=True memakai parser negation agar query seperti "without patents" menghasilkan excluded_terms.
# negation=False/NaN memakai HNSW murni tanpa parser negation.
SEARCH_MODE_BY_NEGATION = {
    True: "negation",
    False: "no_negation",
    None: "no_negation",
}

SEARCH_ENGINE_CONFIGS = {
    "no_negation": {
        "use_reranker": False,
        "use_llm": False,
        "use_finetuned": False,
    },
    "negation": {
        "use_reranker": False,
        "use_llm": False,
        "use_finetuned": True,
    },
}

collection_plan_df = pd.DataFrame(COLLECTION_CONFIGS)
print("Collections to evaluate:")
display(collection_plan_df)
print("Search hnsw_ef values:", EF_VALUES)
print("Qdrant return limits:", QDRANT_LIMIT_VALUES)

Collections to evaluate:


,collection_id,collection_name,m,ef_construct
0,m8_ef100,rtas_hybrid_m8_ef100,8,100
1,m8_ef200,rtas_hybrid_m8_ef200,8,200
2,m8_ef400,rtas_hybrid_m8_ef400,8,400
3,m16_ef100,rtas_hybrid_m16_ef100,16,100
4,m16_ef200,rtas_hybrid_m16_ef200,16,200
5,m16_ef400,rtas_hybrid_m16_ef400,16,400
6,m32_ef100,rtas_hybrid_m32_ef100,32,100
7,m32_ef200,rtas_hybrid_m32_ef200,32,200
8,m32_ef400,rtas_hybrid_m32_ef400,32,400


Search hnsw_ef values: [16, 32, 64, 128, 256]
Qdrant return limits: [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]


## Helper: Normalisasi, Ranking, dan NDCG

In [42]:
def parse_bool_like(value: Any) -> bool | None:
    if pd.isna(value):
        return None
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n"}:
        return False
    return None


def parse_rank(rank_str: Any) -> tuple[int, int] | None:
    """Parse rank_qdrant as (agreement_rank, article_rank). Example: '1,2' = agreement top 1, article top 2."""
    if pd.isna(rank_str) or str(rank_str).strip() == "":
        return None
    try:
        parts = str(rank_str).replace('"', "").replace("'", "").strip().split(",")
        return int(float(parts[0])), int(float(parts[1]))
    except Exception:
        return None


def normalize_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    text = str(value)
    text = text.replace("â", "'").replace("’", "'").replace(" ", " ")
    text = re.sub(r"\bartical\b", "article", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text).strip().casefold()
    return text


def normalize_rta_id(value: Any) -> str:
    if pd.isna(value) or str(value).strip() == "":
        return ""
    try:
        return str(int(float(value)))
    except Exception:
        return str(value).strip()


def make_rta_key(rta_id: Any, article_number: Any, article_title: Any) -> tuple[str, str, str, str] | None:
    rid = normalize_rta_id(rta_id)
    if not rid:
        return None
    return ("rta", rid, normalize_text(article_number), normalize_text(article_title))


def make_agreement_key(agreement_title: Any, article_number: Any, article_title: Any) -> tuple[str, str, str, str]:
    return (
        "agreement",
        normalize_text(agreement_title),
        normalize_text(article_number),
        normalize_text(article_title),
    )


def compute_dcg(scores: list[float]) -> float:
    return sum(rel / np.log2(idx + 2) for idx, rel in enumerate(scores))


def compute_ndcg(predicted_relevance: list[float], ideal_relevance: list[float], k: int = K) -> float:
    scores = list(predicted_relevance[:k])
    if len(scores) < k:
        scores.extend([0] * (k - len(scores)))
    ideal = sorted(list(ideal_relevance), reverse=True)[:k]
    idcg = compute_dcg(ideal)
    return compute_dcg(scores) / idcg if idcg > 0 else 0.0

## Load Ground Truth

Kolom `negation` dipakai untuk memisahkan Q02 menjadi dua unit evaluasi (`Q02_negation` dan `Q02_no_negation`) karena CSV berisi dua pool hasil untuk query yang sama.

In [43]:
def load_ground_truth(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required_columns = {
        "query",
        "query_id",
        "rta_id",
        "agreement_title",
        "chapter_title",
        "article_number",
        "article_title",
        "article_text",
        "rank_qdrant",
        "relevance",
    }
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"Missing GT columns: {sorted(missing)}")

    df = df.copy()
    df["relevance"] = pd.to_numeric(df["relevance"], errors="coerce").fillna(0).astype(float)
    df["negation_flag"] = df.get("negation", pd.Series([None] * len(df))).apply(parse_bool_like)

    def make_eval_id(row: pd.Series) -> str:
        if row["negation_flag"] is True:
            return f"{row['query_id']}_negation"
        if row["negation_flag"] is False:
            return f"{row['query_id']}_no_negation"
        return str(row["query_id"])

    df["eval_id"] = df.apply(make_eval_id, axis=1)
    rank_parts = df["rank_qdrant"].apply(parse_rank)
    df["parsed_rank_qdrant"] = rank_parts
    df["agreement_rank_qdrant"] = rank_parts.apply(lambda rank: rank[0] if rank else np.nan)
    df["article_rank_qdrant"] = rank_parts.apply(lambda rank: rank[1] if rank else np.nan)
    return df


gt_df = load_ground_truth(GT_PATH)
print("GT rows:", len(gt_df))
display(gt_df.groupby(["eval_id", "query_id", "query"], dropna=False).size().reset_index(name="gt_rows"))

GT rows: 108


,eval_id,query_id,query,gt_rows
0,Q01,Q01,customs clearance for perishable goods,29
1,Q02_negation,Q02,intellectual property protection without patents,25
2,Q02_no_negation,Q02,intellectual property protection without patents,26
3,Q03,Q03,labour standards in trade,28


In [44]:
def build_gt_lookup(df_unit: pd.DataFrame) -> dict[tuple[str, str, str, str], dict[str, Any]]:
    lookup = {}
    for idx, row in df_unit.iterrows():
        record = {
            "gt_index": idx,
            "relevance": float(row["relevance"]),
            "rta_id": row.get("rta_id"),
            "agreement_title": row.get("agreement_title"),
            "chapter_title": row.get("chapter_title"),
            "article_number": row.get("article_number"),
            "article_title": row.get("article_title"),
        }

        keys = [make_agreement_key(row.get("agreement_title"), row.get("article_number"), row.get("article_title"))]
        rta_key = make_rta_key(row.get("rta_id"), row.get("article_number"), row.get("article_title"))
        if rta_key is not None:
            keys.append(rta_key)

        for key in keys:
            # Jika ada duplikasi key dalam satu eval unit, pakai relevance tertinggi.
            if key not in lookup or record["relevance"] > lookup[key]["relevance"]:
                lookup[key] = record
    return lookup


def prediction_candidate_keys(item: dict[str, Any]) -> list[tuple[str, str, str, str]]:
    keys = [make_agreement_key(item.get("agreement"), item.get("article_number"), item.get("article_name"))]
    rta_key = make_rta_key(item.get("rta_id"), item.get("article_number"), item.get("article_name"))
    if rta_key is not None:
        keys.insert(0, rta_key)
    return keys


def find_prediction_in_gt(item: dict[str, Any], gt_lookup: dict[tuple[str, str, str, str], dict[str, Any]]):
    for key in prediction_candidate_keys(item):
        if key in gt_lookup:
            return key, gt_lookup[key]
    return None, None

## Baseline dari `rank_qdrant` di GT

Cell ini hanya membaca ranking yang sudah tersimpan di CSV, supaya hasil HNSW nanti punya pembanding awal.

`rank_qdrant` dibaca sebagai `(agreement_rank, article_rank)`. Contoh: `1,2` berarti agreement urutan 1 dan artikel urutan 2 dalam agreement tersebut.

In [45]:
def evaluate_gt_rank_column(gt: pd.DataFrame, k: int = K) -> pd.DataFrame:
    rows = []
    for eval_id, df_unit in gt.groupby("eval_id", sort=True):
        ranked = df_unit.dropna(subset=["parsed_rank_qdrant"]).copy()
        ranked = ranked.sort_values(["agreement_rank_qdrant", "article_rank_qdrant"])
        relevance_sequence = ranked["relevance"].tolist()[:k]
        ndcg = compute_ndcg(relevance_sequence, df_unit["relevance"].tolist(), k=k)
        rows.append(
            {
                "eval_id": eval_id,
                "query_id": df_unit["query_id"].iloc[0],
                "query": df_unit["query"].iloc[0],
                "gt_rows": len(df_unit),
                "ranked_rows": len(ranked),
                "rank_semantics": "agreement_rank,article_rank",
                f"ndcg_at_{k}": ndcg,
                "relevance_sequence": relevance_sequence,
            }
        )
    return pd.DataFrame(rows)


baseline_df = evaluate_gt_rank_column(gt_df, k=K)
display(baseline_df[["eval_id", "gt_rows", "ranked_rows", f"ndcg_at_{K}"]])
print("Mean baseline NDCG:", baseline_df[f"ndcg_at_{K}"].mean())

,eval_id,gt_rows,ranked_rows,ndcg_at_30
0,Q01,29,29,0.898623
1,Q02_negation,25,25,0.900094
2,Q02_no_negation,26,26,0.885123
3,Q03,28,28,0.923910


Mean baseline NDCG: 0.9019376156062544


## HNSW Search yang Mengikuti `SearchService`

Fungsi di bawah meniru bagian penting `SearchService.search`: query preprocessing, optional negation filtering, dedup chunk menjadi artikel, grouping per agreement, lalu flatten top 3 artikel dari 10 agreement teratas.

In [46]:
from qdrant_client import models
from qdrant_client.models import FieldCondition, Filter, MatchValue
import src.search_engine as search_engine_module

SearchService = search_engine_module.SearchService
localspelling = search_engine_module.localspelling


def prepare_query(engine: SearchService, user_query: str):
    excluded_terms = []
    llm_iso = None

    if engine.use_finetuned and engine.finetuned_parser:
        parsed = engine.finetuned_parser.parse(user_query)
        search_query = parsed["positive_query"]
        excluded_terms = parsed["excluded_terms"]
    elif engine.use_llm:
        parsed = engine.llm_query_parser(user_query)
        search_query = parsed["positive_query"]
        excluded_terms = parsed["excluded_terms"]
        llm_iso = parsed["iso_code"]
    else:
        search_query = user_query

    detected_iso = llm_iso or engine.detect_iso_from_query(search_query)
    qdrant_filter = None
    if detected_iso:
        qdrant_filter = Filter(
            must=[FieldCondition(key="iso", match=MatchValue(value=detected_iso))]
        )

    if not engine.use_finetuned:
        words = search_query.split()
        misspelled = engine.spell.unknown(words)
        corrected = []
        for word in words:
            if word in misspelled:
                correction = engine.spell.correction(word)
                corrected.append(correction if correction else word)
            else:
                corrected.append(word)
        search_query = " ".join(corrected)

    final_query = localspelling.convert_spelling(search_query, "us")
    excluded_terms = [localspelling.convert_spelling(term, "us") for term in excluded_terms]
    return final_query, excluded_terms, qdrant_filter


def build_result_from_hit(hit) -> dict[str, Any]:
    payload = hit.payload or {}
    article_text = payload.get("article_text") or "-"
    article_text_display = SearchService._resolve_article_text_display(
        payload.get("article_text_display"), article_text
    )
    score = float(hit.score)
    return {
        "id": hit.id,
        "rta_id": payload.get("rta_id"),
        "agreement": payload.get("Agreement") or "-",
        "chapter_name": payload.get("chapter_name") or "-",
        "article_id": payload.get("article_id"),
        "article_number": payload.get("article_number") or "-",
        "article_name": payload.get("article_name") or "-",
        "article_text": article_text,
        "article_text_display": article_text_display,
        "acronym": payload.get("acronym") or "-",
        "iso": payload.get("iso") or "-",
        "score": score,
        "confidence": f"{score:.4%}",
    }


def dedupe_hits_like_search_engine(hits) -> list[Any]:
    unique_hits = []
    seen_articles = set()
    for hit in hits:
        payload = hit.payload or {}
        unique_key = f"{payload.get('rta_id')}_{str(payload.get('article_name', '')).strip()}"
        if unique_key not in seen_articles:
            seen_articles.add(unique_key)
            unique_hits.append(hit)
    return unique_hits


def apply_negation_filter(results: list[dict[str, Any]], excluded_terms: list[str]) -> list[dict[str, Any]]:
    if not excluded_terms:
        return results
    filtered = []
    for item in results:
        text = item.get("article_text_display", "").lower()
        if not any(term.lower() in text for term in excluded_terms):
            filtered.append(item)
    return filtered


def group_by_agreement(results: list[dict[str, Any]], agreement_limit: int = AGREEMENT_LIMIT) -> list[dict[str, Any]]:
    grouped = {}
    for item in sorted(results, key=lambda x: x["score"], reverse=True):
        rta_id = item["rta_id"]
        if rta_id not in grouped:
            grouped[rta_id] = {
                "rta_id": rta_id,
                "agreement": item["agreement"],
                "acronym": item["acronym"],
                "iso": item["iso"],
                "best_score": item["score"],
                "best_confidence": item["confidence"],
                "articles": [],
            }
        grouped[rta_id]["articles"].append(
            {
                "id": item["id"],
                "chapter_name": item["chapter_name"],
                "article_id": item["article_id"],
                "article_number": item["article_number"],
                "article_name": item["article_name"],
                "article_text": item["article_text"],
                "article_text_display": item["article_text_display"],
                "score": item["score"],
                "confidence": item["confidence"],
            }
        )
    return sorted(grouped.values(), key=lambda x: x["best_score"], reverse=True)[:agreement_limit]


def flatten_grouped_results(grouped_results: list[dict[str, Any]], top_articles: int = TOP_ARTICLES_PER_AGREEMENT) -> list[dict[str, Any]]:
    flattened = []
    global_rank = 1
    for agreement_rank, group in enumerate(grouped_results, start=1):
        articles = sorted(group.get("articles", []), key=lambda x: x["score"], reverse=True)[:top_articles]
        for article_rank, article in enumerate(articles, start=1):
            flattened.append(
                {
                    "global_rank": global_rank,
                    "agreement_rank": agreement_rank,
                    "article_rank": article_rank,
                    "rta_id": group.get("rta_id"),
                    "agreement": group.get("agreement"),
                    **article,
                }
            )
            global_rank += 1
    return flattened


def hnsw_search_articles(
    engine: SearchService,
    query: str,
    hnsw_ef: int,
    collection_name: str,
    raw_limit: int,
    agreement_limit: int = AGREEMENT_LIMIT,
    top_articles: int = TOP_ARTICLES_PER_AGREEMENT,
    threshold: float = THRESHOLD,
) -> tuple[list[dict[str, Any]], float, str, int, int]:
    final_query, excluded_terms, qdrant_filter = prepare_query(engine, query)
    query_vector = engine.vector_model.encode(final_query).tolist()

    started = time.perf_counter()
    raw_hits = engine.client.query_points(
        collection_name=collection_name,
        query=query_vector,
        query_filter=qdrant_filter,
        limit=raw_limit,
        search_params=models.SearchParams(hnsw_ef=hnsw_ef),
        with_payload=True,
    ).points
    elapsed_ms = (time.perf_counter() - started) * 1000

    unique_hits = dedupe_hits_like_search_engine(raw_hits)
    results = [build_result_from_hit(hit) for hit in unique_hits]
    results = [item for item in results if item["score"] >= threshold]
    results = apply_negation_filter(results, excluded_terms)
    grouped = group_by_agreement(results, agreement_limit=agreement_limit)
    flattened = flatten_grouped_results(grouped, top_articles=top_articles)
    return flattened, elapsed_ms, final_query, len(raw_hits), len(unique_hits)

## Evaluasi HNSW terhadap Ground Truth

In [47]:
def relevance_from_predictions(
    predictions: list[dict[str, Any]],
    gt_lookup: dict[tuple[str, str, str, str], dict[str, Any]],
    filter_to_gt_pool: bool = FILTER_TO_GT_POOL,
    k: int = K,
) -> tuple[list[float], list[dict[str, Any]]]:
    relevance_sequence = []
    matched_rows = []
    seen_gt_keys = set()

    for prediction in predictions:
        gt_key, gt_record = find_prediction_in_gt(prediction, gt_lookup)

        if gt_record is None:
            if filter_to_gt_pool:
                continue
            relevance = 0.0
        else:
            if gt_key in seen_gt_keys:
                continue
            seen_gt_keys.add(gt_key)
            relevance = float(gt_record["relevance"])

        relevance_sequence.append(relevance)
        matched_rows.append(
            {
                "predicted_global_rank": prediction.get("global_rank"),
                "predicted_agreement_rank": prediction.get("agreement_rank"),
                "predicted_article_rank": prediction.get("article_rank"),
                "rta_id": prediction.get("rta_id"),
                "agreement": prediction.get("agreement"),
                "chapter_name": prediction.get("chapter_name"),
                "article_number": prediction.get("article_number"),
                "article_name": prediction.get("article_name"),
                "score": prediction.get("score"),
                "relevance": relevance,
                "matched_in_gt": gt_record is not None,
            }
        )

        if len(relevance_sequence) >= k:
            break

    return relevance_sequence, matched_rows


def get_unit_negation_flag(df_unit: pd.DataFrame) -> bool | None:
    flags = [flag for flag in df_unit["negation_flag"].dropna().unique().tolist()]
    if not flags:
        return None
    if len(flags) > 1:
        raise ValueError(f"One eval unit has mixed negation flags: {flags}")
    return bool(flags[0])


def get_search_mode(negation_flag: bool | None) -> str:
    return SEARCH_MODE_BY_NEGATION.get(negation_flag, "no_negation")


def get_search_engine_for_mode(
    search_mode: str,
    engine_cache: dict[str, SearchService],
) -> SearchService:
    if search_mode not in engine_cache:
        config = SEARCH_ENGINE_CONFIGS[search_mode]
        engine_cache[search_mode] = SearchService(**config)
    return engine_cache[search_mode]


def evaluate_hnsw_configs(
    gt: pd.DataFrame,
    collection_configs: list[dict[str, Any]] = COLLECTION_CONFIGS,
    ef_values: list[int] = EF_VALUES,
    qdrant_limit_values: list[int] = QDRANT_LIMIT_VALUES,
    k: int = K,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    summary_rows = []
    detail_rows = []
    search_cache = {}
    engine_cache = {}

    total_jobs = len(collection_configs) * len(ef_values) * len(qdrant_limit_values) * gt["eval_id"].nunique()
    done_jobs = 0

    for collection_config in collection_configs:
        collection_id = collection_config["collection_id"]
        collection_name = collection_config["collection_name"]
        print(f"Evaluating collection {collection_id} ({collection_name})")

        for hnsw_ef in ef_values:
            for qdrant_limit in qdrant_limit_values:
                for eval_id, df_unit in gt.groupby("eval_id", sort=True):
                    done_jobs += 1
                    query = df_unit["query"].iloc[0]
                    negation_flag = get_unit_negation_flag(df_unit)
                    search_mode = get_search_mode(negation_flag)
                    engine = get_search_engine_for_mode(search_mode, engine_cache)
                    cache_key = (collection_name, query, hnsw_ef, qdrant_limit, search_mode)

                    if cache_key not in search_cache:
                        predictions, elapsed_ms, final_query, raw_hit_count, unique_hit_count = hnsw_search_articles(
                            engine,
                            query,
                            hnsw_ef=hnsw_ef,
                            collection_name=collection_name,
                            raw_limit=qdrant_limit,
                        )
                        search_cache[cache_key] = (
                            predictions,
                            elapsed_ms,
                            final_query,
                            raw_hit_count,
                            unique_hit_count,
                        )
                    else:
                        predictions, elapsed_ms, final_query, raw_hit_count, unique_hit_count = search_cache[cache_key]

                    gt_lookup = build_gt_lookup(df_unit)
                    relevance_sequence, matched_rows = relevance_from_predictions(
                        predictions,
                        gt_lookup,
                        filter_to_gt_pool=FILTER_TO_GT_POOL,
                        k=k,
                    )
                    ndcg = compute_ndcg(relevance_sequence, df_unit["relevance"].tolist(), k=k)

                    summary_rows.append(
                        {
                            "collection_id": collection_id,
                            "collection_name": collection_name,
                            "m": collection_config["m"],
                            "ef_construct": collection_config["ef_construct"],
                            "hnsw_ef": hnsw_ef,
                            "qdrant_limit": qdrant_limit,
                            "eval_id": eval_id,
                            "query_id": df_unit["query_id"].iloc[0],
                            "query": query,
                            "negation_flag": negation_flag,
                            "search_mode": search_mode,
                            "engine_use_finetuned": engine.use_finetuned,
                            "engine_use_llm": engine.use_llm,
                            "engine_use_reranker": engine.use_reranker,
                            "final_query": final_query,
                            "gt_rows": len(df_unit),
                            "raw_hit_count": raw_hit_count,
                            "unique_hit_count": unique_hit_count,
                            "predicted_rows_before_gt_filter": len(predictions),
                            "predicted_rows_used": len(relevance_sequence),
                            "elapsed_ms_single_run": elapsed_ms,
                            f"ndcg_at_{k}": ndcg,
                            "relevance_sequence": relevance_sequence,
                        }
                    )

                    for rank_after_filter, row in enumerate(matched_rows, start=1):
                        detail_rows.append(
                            {
                                "collection_id": collection_id,
                                "collection_name": collection_name,
                                "m": collection_config["m"],
                                "ef_construct": collection_config["ef_construct"],
                                "hnsw_ef": hnsw_ef,
                                "qdrant_limit": qdrant_limit,
                                "eval_id": eval_id,
                                "negation_flag": negation_flag,
                                "search_mode": search_mode,
                                "rank_after_gt_filter": rank_after_filter,
                                **row,
                            }
                        )

                    if done_jobs % 20 == 0 or done_jobs == total_jobs:
                        print(f"  progress {done_jobs}/{total_jobs}")

    return pd.DataFrame(summary_rows), pd.DataFrame(detail_rows)

## Benchmark Latency HNSW

Latency benchmark dibuat terpisah dari evaluasi NDCG. Ini mengulang query Qdrant beberapa kali dengan `with_payload=False` agar fokus mengukur efek `hnsw_ef` pada search Qdrant.

In [48]:
def make_latency_specs(gt: pd.DataFrame) -> list[dict[str, Any]]:
    specs = []
    seen = set()
    for eval_id, df_unit in gt.groupby("eval_id", sort=True):
        query = df_unit["query"].iloc[0]
        negation_flag = get_unit_negation_flag(df_unit)
        search_mode = get_search_mode(negation_flag)
        key = (query, search_mode)
        if key in seen:
            continue
        seen.add(key)
        specs.append(
            {
                "eval_id": eval_id,
                "query": query,
                "negation_flag": negation_flag,
                "search_mode": search_mode,
            }
        )
    return specs


def benchmark_hnsw_latency(
    query_specs: list[dict[str, Any]],
    collection_configs: list[dict[str, Any]] = COLLECTION_CONFIGS,
    ef_values: list[int] = EF_VALUES,
    qdrant_limit_values: list[int] = QDRANT_LIMIT_VALUES,
    runs: int = TIMING_RUNS,
) -> pd.DataFrame:
    rows = []
    engine_cache = {}

    for collection_config in collection_configs:
        collection_id = collection_config["collection_id"]
        collection_name = collection_config["collection_name"]
        print(f"Benchmark latency collection {collection_id} ({collection_name})")

        for spec in query_specs:
            query = spec["query"]
            search_mode = spec["search_mode"]
            engine = get_search_engine_for_mode(search_mode, engine_cache)
            final_query, _, qdrant_filter = prepare_query(engine, query)
            query_vector = engine.vector_model.encode(final_query).tolist()

            # Warmup
            engine.client.query_points(
                collection_name=collection_name,
                query=query_vector,
                query_filter=qdrant_filter,
                limit=1,
                with_payload=False,
            )

            for hnsw_ef in ef_values:
                for qdrant_limit in qdrant_limit_values:
                    times = []
                    for _ in range(runs):
                        started = time.perf_counter()
                        engine.client.query_points(
                            collection_name=collection_name,
                            query=query_vector,
                            query_filter=qdrant_filter,
                            limit=qdrant_limit,
                            search_params=models.SearchParams(hnsw_ef=hnsw_ef),
                            with_payload=False,
                        )
                        times.append((time.perf_counter() - started) * 1000)

                    rows.append(
                        {
                            "collection_id": collection_id,
                            "collection_name": collection_name,
                            "m": collection_config["m"],
                            "ef_construct": collection_config["ef_construct"],
                            "eval_id": spec["eval_id"],
                            "query": query,
                            "negation_flag": spec["negation_flag"],
                            "search_mode": search_mode,
                            "engine_use_finetuned": engine.use_finetuned,
                            "engine_use_llm": engine.use_llm,
                            "final_query": final_query,
                            "hnsw_ef": hnsw_ef,
                            "qdrant_limit": qdrant_limit,
                            "runs": runs,
                            "avg_time_ms": float(np.mean(times)),
                            "min_time_ms": float(np.min(times)),
                            "max_time_ms": float(np.max(times)),
                            "std_time_ms": float(np.std(times)),
                        }
                    )
    return pd.DataFrame(rows)

## Jalankan Evaluasi

Jalankan cell ini saat Qdrant dan semua collection hasil indexing sudah aktif. Output lengkap disimpan ke `evaluation/hnsw_eval_results/`, termasuk ringkasan final untuk melihat konfigurasi terbaik secara cepat.

In [49]:
hnsw_summary_df, hnsw_detail_df = evaluate_hnsw_configs(
    gt_df,
    collection_configs=COLLECTION_CONFIGS,
    ef_values=EF_VALUES,
    qdrant_limit_values=QDRANT_LIMIT_VALUES,
    k=K,
)

summary_cols = [
    "collection_id",
    "m",
    "ef_construct",
    "hnsw_ef",
    "qdrant_limit",
    "eval_id",
    "negation_flag",
    "search_mode",
    "engine_use_finetuned",
    "gt_rows",
    "raw_hit_count",
    "unique_hit_count",
    "predicted_rows_before_gt_filter",
    "predicted_rows_used",
    "elapsed_ms_single_run",
    f"ndcg_at_{K}",
]
display(hnsw_summary_df[summary_cols].sort_values(["collection_id", "hnsw_ef", "qdrant_limit", "eval_id"]))

grid_preview_df = (
    hnsw_summary_df.groupby(["collection_id", "m", "ef_construct", "hnsw_ef", "qdrant_limit"])
    .agg(
        mean_ndcg=(f"ndcg_at_{K}", "mean"),
        min_ndcg=(f"ndcg_at_{K}", "min"),
        mean_elapsed_ms=("elapsed_ms_single_run", "mean"),
        mean_used_rows=("predicted_rows_used", "mean"),
    )
    .reset_index()
    .sort_values(["mean_ndcg", "mean_elapsed_ms"], ascending=[False, True])
)
display(grid_preview_df.head(20))

2026-05-29 01:14:34 - src.search_engine - INFO - [Init] Initializing model & database
2026-05-29 01:14:34 - httpx - INFO - HTTP Request: GET http://127.0.0.1:6333 "HTTP/1.1 200 OK"
2026-05-29 01:14:34 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cuda:0
2026-05-29 01:14:34 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-mpnet-base-v2


Evaluating collection m8_ef100 (rtas_hybrid_m8_ef100)


2026-05-29 01:14:34 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 01:14:34 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
2026-05-29 01:14:35 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 01:14:35 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-29 01:14:35 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "H

  progress 20/1800


Batches: 100%|██████████| 1/1 [00:00<00:00, 103.12it/s]
2026-05-29 01:14:48 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef100/points/query "HTTP/1.1 200 OK"
2026-05-29 01:14:48 - finetuning_negation.FLAN.inference - INFO - [NegationParser] Query   : 'intellectual property protection without patents'
2026-05-29 01:14:48 - finetuning_negation.FLAN.inference - INFO - [NegationParser] Raw out : 'positive: intellectual property protection | excluded: patents | country: none'
2026-05-29 01:14:48 - finetuning_negation.FLAN.inference - INFO - [NegationParser] Parsed  : {'positive_query': 'intellectual property protection', 'excluded_terms': ['patents'], 'iso_code': None, 'confidence': 1.0, 'raw': 'positive: intellectual property protection | excluded: patents | country: none'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 104.81it/s]
2026-05-29 01:14:48 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef100/points/query

...

  progress 1780/1800


Batches: 100%|██████████| 1/1 [00:00<00:00, 117.99it/s]
2026-05-29 01:17:08 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef400/points/query "HTTP/1.1 200 OK"
2026-05-29 01:17:08 - finetuning_negation.FLAN.inference - INFO - [NegationParser] Query   : 'intellectual property protection without patents'
2026-05-29 01:17:08 - finetuning_negation.FLAN.inference - INFO - [NegationParser] Raw out : 'positive: intellectual property protection | excluded: patents | country: none'
2026-05-29 01:17:08 - finetuning_negation.FLAN.inference - INFO - [NegationParser] Parsed  : {'positive_query': 'intellectual property protection', 'excluded_terms': ['patents'], 'iso_code': None, 'confidence': 1.0, 'raw': 'positive: intellectual property protection | excluded: patents | country: none'}
Batches: 100%|██████████| 1/1 [00:00<00:00, 124.53it/s]
2026-05-29 01:17:08 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef400/points/que

  progress 1800/1800


,collection_id,m,ef_construct,hnsw_ef,qdrant_limit,eval_id,negation_flag,search_mode,engine_use_finetuned,gt_rows,raw_hit_count,unique_hit_count,predicted_rows_before_gt_filter,predicted_rows_used,elapsed_ms_single_run,ndcg_at_30
600,m16_ef100,16,100,16,10,Q01,None,no_negation,False,29,10,10,10,9,737.257931,0.654943
601,m16_ef100,16,100,16,10,Q02_negation,True,negation,True,25,10,10,3,2,24.926887,0.196040
602,m16_ef100,16,100,16,10,Q02_no_negation,False,no_negation,False,26,10,10,3,2,6.103338,0.238238
603,m16_ef100,16,100,16,10,Q03,None,no_negation,False,28,10,10,9,8,23.932580,0.454744
604,m16_ef100,16,100,16,20,Q01,None,no_negation,False,29,20,20,10,9,10.945953,0.654943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,m8_ef400,8,400,256,90,Q03,None,no_negation,False,28,90,85,22,19,12.774104,0.836720
596,m8_ef400,8,400,256,100,Q01,None,no_negation,False,29,100,97,17,16,13.388570,0.779092
597,m8_ef400,8,400,256,100,Q02_negation,True,negation,True,25,100,100,16,16,10.703452,0.702977
598,m8_ef400,8,400,256,100,Q02_no_negation,False,no_negation,False,26,100,100,17,15,11.366992,0.718324


,collection_id,m,ef_construct,hnsw_ef,qdrant_limit,mean_ndcg,min_ndcg,mean_elapsed_ms,mean_used_rows
449,m8_ef400,8,400,256,100,0.759278,0.702977,12.049763,16.50
89,m16_ef200,16,200,128,100,0.754368,0.657336,10.752022,16.25
129,m16_ef400,16,400,64,100,0.752622,0.676352,13.113561,16.25
139,m16_ef400,16,400,128,100,0.751237,0.644809,11.878017,16.25
439,m8_ef400,8,400,128,100,0.751129,0.670381,11.261474,16.25
199,m32_ef100,32,100,256,100,0.750912,0.669514,12.001590,16.25
79,m16_ef200,16,200,64,100,0.749935,0.691368,9.965631,16.25
249,m32_ef200,32,200,256,100,0.748789,0.661020,13.022440,16.00
149,m16_ef400,16,400,256,100,0.745883,0.676352,12.970953,16.25
279,m32_ef400,32,400,64,100,0.745109,0.646302,11.046789,16.25


In [50]:
# Optional: benchmark latency murni Qdrant.
# Q02 akan dihitung dua kali karena mode negation dan no_negation memang berbeda.
latency_specs = make_latency_specs(gt_df)
latency_df = benchmark_hnsw_latency(
    latency_specs,
    collection_configs=COLLECTION_CONFIGS,
    ef_values=EF_VALUES,
    qdrant_limit_values=QDRANT_LIMIT_VALUES,
    runs=TIMING_RUNS,
)

display(latency_df)
display(
    latency_df.groupby(["collection_id", "hnsw_ef", "qdrant_limit", "search_mode"])
    .agg(
        avg_time_ms=("avg_time_ms", "mean"),
        min_time_ms=("min_time_ms", "min"),
        max_time_ms=("max_time_ms", "max"),
        std_time_ms=("std_time_ms", "mean"),
    )
    .reset_index()
    .sort_values(["collection_id", "hnsw_ef", "qdrant_limit", "search_mode"])
)

2026-05-29 01:17:27 - src.search_engine - INFO - [Init] Initializing model & database
2026-05-29 01:17:27 - httpx - INFO - HTTP Request: GET http://127.0.0.1:6333 "HTTP/1.1 200 OK"
2026-05-29 01:17:27 - sentence_transformers.SentenceTransformer - INFO - Use pytorch device_name: cuda:0
2026-05-29 01:17:27 - sentence_transformers.SentenceTransformer - INFO - Load pretrained SentenceTransformer: all-mpnet-base-v2


Benchmark latency collection m8_ef100 (rtas_hybrid_m8_ef100)


2026-05-29 01:17:28 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 01:17:28 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"
2026-05-29 01:17:28 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 01:17:28 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-29 01:17:28 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "H

Benchmark latency collection m8_ef200 (rtas_hybrid_m8_ef200)


Batches: 100%|██████████| 1/1 [00:00<00:00, 82.10it/s]
2026-05-29 01:18:17 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:18:17 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:18:17 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:18:17 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:18:17 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:18:17 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m8_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:18:17 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rta

...

Batches: 100%|██████████| 1/1 [00:00<00:00, 121.27it/s]
2026-05-29 01:21:58 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:21:58 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:21:58 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:21:58 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:21:58 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:21:58 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef200/points/query "HTTP/1.1 200 OK"
2026-05-29 01:21:58 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collecti

Benchmark latency collection m32_ef400 (rtas_hybrid_m32_ef400)


Batches: 100%|██████████| 1/1 [00:00<00:00, 116.14it/s]
2026-05-29 01:22:36 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef400/points/query "HTTP/1.1 200 OK"
2026-05-29 01:22:36 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef400/points/query "HTTP/1.1 200 OK"
2026-05-29 01:22:36 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef400/points/query "HTTP/1.1 200 OK"
2026-05-29 01:22:36 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef400/points/query "HTTP/1.1 200 OK"
2026-05-29 01:22:36 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef400/points/query "HTTP/1.1 200 OK"
2026-05-29 01:22:36 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collections/rtas_hybrid_m32_ef400/points/query "HTTP/1.1 200 OK"
2026-05-29 01:22:36 - httpx - INFO - HTTP Request: POST http://127.0.0.1:6333/collecti

,collection_id,collection_name,m,ef_construct,eval_id,query,negation_flag,search_mode,engine_use_finetuned,engine_use_llm,final_query,hnsw_ef,qdrant_limit,runs,avg_time_ms,min_time_ms,max_time_ms,std_time_ms
0,m8_ef100,rtas_hybrid_m8_ef100,8,100,Q01,customs clearance for perishable goods,None,no_negation,False,False,customs clearance for perishable goods,16,10,25,6.740308,4.022620,47.724183,8.384921
1,m8_ef100,rtas_hybrid_m8_ef100,8,100,Q01,customs clearance for perishable goods,None,no_negation,False,False,customs clearance for perishable goods,16,20,25,6.309609,4.449370,24.033643,3.686833
2,m8_ef100,rtas_hybrid_m8_ef100,8,100,Q01,customs clearance for perishable goods,None,no_negation,False,False,customs clearance for perishable goods,16,30,25,5.461916,4.437738,6.601337,0.656137
3,m8_ef100,rtas_hybrid_m8_ef100,8,100,Q01,customs clearance for perishable goods,None,no_negation,False,False,customs clearance for perishable goods,16,40,25,5.452259,4.111198,9.036021,1.055688
4,m8_ef100,rtas_hybrid_m8_ef100,8,100,Q01,customs clearance for perishable goods,None,no_negation,False,False,customs clearance for perishable goods,16,50,25,7.085426,4.425755,15.656744,2.537229
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1795,m32_ef400,rtas_hybrid_m32_ef400,32,400,Q03,labour standards in trade,None,no_negation,False,False,labor standards in trade,256,60,25,10.239112,7.297198,16.201939,2.425622
1796,m32_ef400,rtas_hybrid_m32_ef400,32,400,Q03,labour standards in trade,None,no_negation,False,False,labor standards in trade,256,70,25,11.151490,7.623107,19.825030,2.641349
1797,m32_ef400,rtas_hybrid_m32_ef400,32,400,Q03,labour standards in trade,None,no_negation,False,False,labor standards in trade,256,80,25,10.987043,8.027865,15.958517,2.374410
1798,m32_ef400,rtas_hybrid_m32_ef400,32,400,Q03,labour standards in trade,None,no_negation,False,False,labor standards in trade,256,90,25,9.522007,7.097239,13.915988,1.880927


,collection_id,hnsw_ef,qdrant_limit,search_mode,avg_time_ms,min_time_ms,max_time_ms,std_time_ms
0,m16_ef100,16,10,negation,5.819509,3.260592,10.594630,1.709830
1,m16_ef100,16,10,no_negation,5.528431,3.606960,11.897725,1.278969
2,m16_ef100,16,20,negation,5.578455,4.154109,8.718918,1.300868
3,m16_ef100,16,20,no_negation,5.083528,3.210127,12.133643,0.887286
4,m16_ef100,16,30,negation,5.203278,3.307441,10.938694,1.352130
...,...,...,...,...,...,...,...,...
895,m8_ef400,256,80,no_negation,8.915811,5.900797,17.191057,2.242724
896,m8_ef400,256,90,negation,7.932052,6.105164,12.177887,1.445277
897,m8_ef400,256,90,no_negation,8.827939,6.037525,50.103922,3.889624
898,m8_ef400,256,100,negation,9.421354,6.092190,17.897810,2.789417


In [51]:
def build_final_result(
    summary_df: pd.DataFrame,
    latency_df: pd.DataFrame | None,
    k: int = K,
) -> pd.DataFrame:
    group_cols = [
        "collection_id",
        "collection_name",
        "m",
        "ef_construct",
        "hnsw_ef",
        "qdrant_limit",
    ]

    final_df = (
        summary_df.groupby(group_cols)
        .agg(
            mean_ndcg=(f"ndcg_at_{k}", "mean"),
            min_ndcg=(f"ndcg_at_{k}", "min"),
            max_ndcg=(f"ndcg_at_{k}", "max"),
            mean_eval_search_ms=("elapsed_ms_single_run", "mean"),
            mean_raw_hits=("raw_hit_count", "mean"),
            mean_unique_hits=("unique_hit_count", "mean"),
            mean_predicted_rows=("predicted_rows_before_gt_filter", "mean"),
            mean_used_rows=("predicted_rows_used", "mean"),
        )
        .reset_index()
    )

    if latency_df is not None and not latency_df.empty:
        latency_grid_df = (
            latency_df.groupby(group_cols)
            .agg(
                latency_avg_ms=("avg_time_ms", "mean"),
                latency_min_ms=("min_time_ms", "min"),
                latency_max_ms=("max_time_ms", "max"),
                latency_std_ms=("std_time_ms", "mean"),
            )
            .reset_index()
        )
        final_df = final_df.merge(latency_grid_df, on=group_cols, how="left")
    else:
        final_df["latency_avg_ms"] = np.nan
        final_df["latency_min_ms"] = np.nan
        final_df["latency_max_ms"] = np.nan
        final_df["latency_std_ms"] = np.nan

    final_df = final_df.sort_values(
        ["mean_ndcg", "latency_avg_ms", "mean_eval_search_ms", "qdrant_limit"],
        ascending=[False, True, True, True],
        na_position="last",
    ).reset_index(drop=True)
    final_df.insert(0, "overall_rank", np.arange(1, len(final_df) + 1))
    return final_df


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
latency_result_df = latency_df if "latency_df" in globals() else pd.DataFrame()

baseline_path = OUTPUT_DIR / f"baseline_rank_qdrant_ndcg_at_{K}.csv"
summary_path = OUTPUT_DIR / f"all_collections_hnsw_summary_ndcg_at_{K}.csv"
detail_path = OUTPUT_DIR / f"all_collections_hnsw_detail_ndcg_at_{K}.csv"
latency_path = OUTPUT_DIR / "all_collections_hnsw_latency.csv"
final_grid_path = OUTPUT_DIR / f"final_result_grid_ndcg_at_{K}.csv"
final_best_overall_path = OUTPUT_DIR / f"final_result_best_overall_ndcg_at_{K}.csv"
final_best_per_collection_path = OUTPUT_DIR / f"final_result_best_per_collection_ndcg_at_{K}.csv"

baseline_df.to_csv(baseline_path, index=False)
hnsw_summary_df.to_csv(summary_path, index=False)
hnsw_detail_df.to_csv(detail_path, index=False)
latency_result_df.to_csv(latency_path, index=False)

final_result_df = build_final_result(hnsw_summary_df, latency_result_df, k=K)
final_best_overall_df = final_result_df.head(20).copy()
final_best_per_collection_df = (
    final_result_df.sort_values(["collection_id", "overall_rank"])
    .groupby("collection_id", as_index=False)
    .head(1)
    .sort_values("overall_rank")
)

final_result_df.to_csv(final_grid_path, index=False)
final_best_overall_df.to_csv(final_best_overall_path, index=False)
final_best_per_collection_df.to_csv(final_best_per_collection_path, index=False)

for collection_id, collection_summary_df in hnsw_summary_df.groupby("collection_id"):
    collection_dir = OUTPUT_DIR / collection_id
    collection_dir.mkdir(parents=True, exist_ok=True)

    baseline_df.to_csv(collection_dir / f"baseline_rank_qdrant_ndcg_at_{K}.csv", index=False)
    collection_summary_df.to_csv(collection_dir / f"hnsw_summary_ndcg_at_{K}.csv", index=False)
    hnsw_detail_df[hnsw_detail_df["collection_id"] == collection_id].to_csv(
        collection_dir / f"hnsw_detail_ndcg_at_{K}.csv", index=False
    )
    if not latency_result_df.empty and "collection_id" in latency_result_df.columns:
        collection_latency_df = latency_result_df[latency_result_df["collection_id"] == collection_id]
    else:
        collection_latency_df = latency_result_df
    collection_latency_df.to_csv(collection_dir / "hnsw_latency.csv", index=False)
    final_result_df[final_result_df["collection_id"] == collection_id].to_csv(
        collection_dir / f"final_result_grid_ndcg_at_{K}.csv", index=False
    )

print("Saved main outputs:")
print("-", baseline_path)
print("-", summary_path)
print("-", detail_path)
print("-", latency_path)
print("-", final_grid_path)
print("-", final_best_overall_path)
print("-", final_best_per_collection_path)

print("\nTop final result:")
display(final_best_overall_df)

print("\nBest per collection:")
display(final_best_per_collection_df)


Saved main outputs:
- /home/vinik/Github/vector-search-thesis/evaluation/hnsw_eval_results/baseline_rank_qdrant_ndcg_at_30.csv
- /home/vinik/Github/vector-search-thesis/evaluation/hnsw_eval_results/all_collections_hnsw_summary_ndcg_at_30.csv
- /home/vinik/Github/vector-search-thesis/evaluation/hnsw_eval_results/all_collections_hnsw_detail_ndcg_at_30.csv
- /home/vinik/Github/vector-search-thesis/evaluation/hnsw_eval_results/all_collections_hnsw_latency.csv
- /home/vinik/Github/vector-search-thesis/evaluation/hnsw_eval_results/final_result_grid_ndcg_at_30.csv
- /home/vinik/Github/vector-search-thesis/evaluation/hnsw_eval_results/final_result_best_overall_ndcg_at_30.csv
- /home/vinik/Github/vector-search-thesis/evaluation/hnsw_eval_results/final_result_best_per_collection_ndcg_at_30.csv

Top final result:


,overall_rank,collection_id,collection_name,m,ef_construct,hnsw_ef,qdrant_limit,mean_ndcg,min_ndcg,max_ndcg,mean_eval_search_ms,mean_raw_hits,mean_unique_hits,mean_predicted_rows,mean_used_rows,latency_avg_ms,latency_min_ms,latency_max_ms,latency_std_ms
0,1,m8_ef400,rtas_hybrid_m8_ef400,8,400,256,100,0.759278,0.702977,0.83672,12.049763,100.0,97.75,18.00,16.50,9.508657,6.092190,17.963043,2.536206
1,2,m16_ef200,rtas_hybrid_m16_ef200,16,200,128,100,0.754368,0.657336,0.83672,10.752022,100.0,97.75,18.00,16.25,7.843068,5.580487,23.748651,1.992348
2,3,m16_ef400,rtas_hybrid_m16_ef400,16,400,64,100,0.752622,0.676352,0.83672,13.113561,100.0,97.75,18.00,16.25,6.662223,5.155020,13.566785,1.378197
3,4,m16_ef400,rtas_hybrid_m16_ef400,16,400,128,100,0.751237,0.644809,0.83672,11.878017,100.0,97.75,18.00,16.25,8.167479,5.845411,15.519503,1.711919
4,5,m8_ef400,rtas_hybrid_m8_ef400,8,400,128,100,0.751129,0.670381,0.83672,11.261474,100.0,97.75,18.00,16.25,7.619170,5.376009,12.481183,1.461790
5,6,m32_ef100,rtas_hybrid_m32_ef100,32,100,256,100,0.750912,0.669514,0.83672,12.001590,100.0,97.75,18.00,16.25,9.112087,6.042045,18.678792,1.946830
6,7,m16_ef200,rtas_hybrid_m16_ef200,16,200,64,100,0.749935,0.691368,0.83672,9.965631,100.0,97.75,18.00,16.25,7.411691,4.936765,37.815896,2.576155
7,8,m32_ef200,rtas_hybrid_m32_ef200,32,200,256,100,0.748789,0.661020,0.83672,13.022440,100.0,97.75,18.00,16.00,8.884558,6.006297,19.156611,1.870925
8,9,m16_ef400,rtas_hybrid_m16_ef400,16,400,256,100,0.745883,0.676352,0.83672,12.970953,100.0,97.75,18.00,16.25,9.900054,6.610534,17.611546,2.558752
9,10,m32_ef400,rtas_hybrid_m32_ef400,32,400,64,100,0.745109,0.646302,0.83672,11.046789,100.0,97.75,18.00,16.25,7.119891,5.128961,15.424303,1.714438



Best per collection:


,overall_rank,collection_id,collection_name,m,ef_construct,hnsw_ef,qdrant_limit,mean_ndcg,min_ndcg,max_ndcg,mean_eval_search_ms,mean_raw_hits,mean_unique_hits,mean_predicted_rows,mean_used_rows,latency_avg_ms,latency_min_ms,latency_max_ms,latency_std_ms
0,1,m8_ef400,rtas_hybrid_m8_ef400,8,400,256,100,0.759278,0.702977,0.836720,12.049763,100.0,97.75,18.0,16.50,9.508657,6.092190,17.963043,2.536206
1,2,m16_ef200,rtas_hybrid_m16_ef200,16,200,128,100,0.754368,0.657336,0.836720,10.752022,100.0,97.75,18.0,16.25,7.843068,5.580487,23.748651,1.992348
2,3,m16_ef400,rtas_hybrid_m16_ef400,16,400,64,100,0.752622,0.676352,0.836720,13.113561,100.0,97.75,18.0,16.25,6.662223,5.155020,13.566785,1.378197
5,6,m32_ef100,rtas_hybrid_m32_ef100,32,100,256,100,0.750912,0.669514,0.836720,12.001590,100.0,97.75,18.0,16.25,9.112087,6.042045,18.678792,1.946830
7,8,m32_ef200,rtas_hybrid_m32_ef200,32,200,256,100,0.748789,0.661020,0.836720,13.022440,100.0,97.75,18.0,16.00,8.884558,6.006297,19.156611,1.870925
9,10,m32_ef400,rtas_hybrid_m32_ef400,32,400,64,100,0.745109,0.646302,0.836720,11.046789,100.0,97.75,18.0,16.25,7.119891,5.128961,15.424303,1.714438
17,18,m8_ef100,rtas_hybrid_m8_ef100,8,100,32,100,0.735005,0.605883,0.836720,11.345058,100.0,97.75,18.0,15.75,6.626198,3.984146,12.729214,1.369643
26,27,m16_ef100,rtas_hybrid_m16_ef100,16,100,16,100,0.726210,0.645286,0.810950,13.251517,100.0,97.50,18.0,16.00,6.170706,3.815937,13.945754,1.497663
63,64,m8_ef200,rtas_hybrid_m8_ef200,8,200,64,100,0.712091,0.638838,0.779092,12.565077,100.0,97.50,18.0,15.25,6.738233,4.517740,12.998014,1.294640
